In [1]:
import hanlp

# Pipeline allows blending multiple callable functions no matter they are a rule, a TensorFlow component or a PyTorch
# one. However, it's slower than the MTL framework.
# pos = hanlp.load(hanlp.pretrained.pos.CTB9_POS_ALBERT_BASE)  # In case both tf and torch are used, load tf first.

HanLP = hanlp.pipeline()\
    .append(hanlp.load('CTB9_TOK_ELECTRA_BASE_CRF'), output_key='tok') \
    .append(hanlp.load('CTB9_POS_ELECTRA_SMALL_TF'), output_key='pos') \
    .append(hanlp.load('CTB9_DEP_ELECTRA_SMALL', conll=0), output_key='dep', input_key='tok')

In [14]:
from YSUtils import *
from collections import defaultdict

In [8]:
output = []
for txt_file in list_txt_files("./instruct_txt/"):
    lines = []
    with open(txt_file, "r", encoding="utf-8") as file:
        for line in file:
            if line == '\n' or len(line.strip()) > 128:
                continue
            lines.append(line.strip())
    doc = HanLP(lines)
    
    
    for i, line in enumerate(lines):
        line_json = {}
        line_json["text"] = line.strip()
        line_deps_with_word = []
        for j, dep_couple in enumerate(doc["dep"][i]):
            str_with_word =  doc["pos"][i][j] + ' ' + doc["pos"][i][dep_couple[0] - 1] + ' ' + dep_couple[1] + ' ' + doc["tok"][i][j] + ' ' + doc["tok"][i][dep_couple[0] - 1] + ' ' + dep_couple[1]
            # str_dep = doc["pos/ctb"][i][j] + ' ' + doc["pos/ctb"][i][dep_couple[0] - 1] + ' ' + dep_couple[1] + ' ' + doc["tok/coarse"][i][j] + ' ' + doc["tok/coarse"][i][j] + ' ' + dep_couple[1]
            line_deps_with_word.append(str_with_word)
        line_json["deps_with_word"] = line_deps_with_word
        output.append(line_json)
with open("./dep_feature/instruct_dep.json", "w", encoding="utf-8") as file:
    file.write(json.dumps(output, ensure_ascii=False))

    

In [17]:
feature_count = defaultdict(int)
with open('./dep_feature/origin_dep_feature.json', 'r', encoding='utf-8') as file:
    origin_dep_feature = json.load(file)
for line in origin_dep_feature:
    for dep in line["deps_with_word"]:
        dep_list = dep.split()
        # 我想要将前三个元素组成字符串
        dep_feature = dep_list[0] + " " + dep_list[1] + " " + dep_list[2]
        feature_count[dep_feature] += 1

with open('./dep_feature/instruct_dep_feature.json', 'r', encoding='utf-8') as file:
    instruct_dep_feature = json.load(file)
for line in instruct_dep_feature:
    for dep in line["deps_with_word"]:
        dep_list = dep.split()
        # 我想要将前三个元素组成字符串
        dep_feature = dep_list[0] + " " + dep_list[1] + " " + dep_list[2]
        feature_count[dep_feature] += 1

sorted_feature_count = dict(sorted(feature_count.items(), key=lambda x: x[1], reverse=True))
with open('./dep_feature/dep_feature_count.json', 'w', encoding='utf-8') as file:
    file.write(json.dumps(sorted_feature_count, ensure_ascii=False))